# Lichess variants: the order of each player's games across game types (free, on Google Colab)

The variants run numbered games separately within each game type, so we cannot yet tell whether a player's standard game came right after an antichess game. This notebook re-reads **only the game headers** for the same 400 players and adds, for every (player, game type, game): its position in the player's overall sequence, the minutes since their previous game of any type, which type that previous game was, and the session it belongs to (new session after 30 minutes without playing).

- **No dates and no names** are saved: only the order, gaps in minutes and anonymous codes. The intermediate files that contain timestamps stay in a private working folder in your Drive and are deleted at the end.
- **Secret key:** the same `LICHESS_ID_SALT` as before (key icon in the left sidebar); the fingerprint is checked.
- It needs the full variants files in `compression-learner-data/lichess_variants/` (already there).
- **Time:** a few hours (it streams both months of all five game types again, but reads only headers). Resumable: if it stops, Run all again.
- **How to run:** Runtime → Run all, allow Drive access.

Output: `compression-learner-data/lichess_variants/game_order_2018-06_2018-07.parquet`

In [ ]:
# Settings
MONTHS = ["2018-06", "2018-07"]
TYPES = ["standard", "chess960", "crazyhouse", "atomic", "antichess"]
EXPECTED = {"standard": 291_436, "chess960": 17_267, "crazyhouse": 35_118, "atomic": 32_389, "antichess": 28_131}  # games kept by the variants run

In [ ]:
# Install tools and fetch the code from the public repo
!apt-get -qq install -y zstd > /dev/null
!pip -q install pyarrow
import os, getpass, subprocess, hashlib, shutil, glob, re
BASE = "https://raw.githubusercontent.com/saurabh9729/compression-learner-data/main/pipelines/lichess"
os.makedirs("/content/work", exist_ok=True); os.chdir("/content/work")
!curl -sSfLO {BASE}/game_order.py
!ls -l *.py

In [ ]:
# Secret key (same as the Lichess run) + fingerprint check, so codes match the existing files
from google.colab import drive
drive.mount("/content/drive")
root = "/content/drive/MyDrive/compression-learner-data"
dest = f"{root}/lichess_variants"
key = None
try:
    from google.colab import userdata
    key = userdata.get("LICHESS_ID_SALT")
except Exception:
    pass
if not key:
    key = getpass.getpass("Secret key: ")
fp = hashlib.pbkdf2_hmac("sha256", key.encode(), b"lichess-key-check", 200_000).hex()[:6]
saved = open(f"{root}/lichess/key_fingerprint.txt").read().strip()
assert fp == saved, f"STOP: not the same key as the Lichess run (fingerprint {fp}, saved {saved})"
print("key check: same key as the Lichess run ✓", fp)
os.environ["LICHESS_ID_SALT"] = key

def url(t, m):
    return (f"https://database.lichess.org/standard/lichess_db_standard_rated_{m}.pgn.zst" if t == "standard"
            else f"https://database.lichess.org/{t}/lichess_db_{t}_rated_{m}.pgn.zst")

In [ ]:
# The 400 players' anonymous codes, taken from the existing variants files
codes = set()
for t in TYPES:
    f = f"{dest}/moves_2018-06_2018-07_{t}_players.csv"
    codes |= {l.split(",")[0] for l in open(f).read().splitlines()[1:] if l}
codes = {c for c in codes if re.fullmatch(r"p_[0-9a-f]{12}", c)}
print("players:", len(codes))
assert len(codes) == 400, "expected 400 players"
open("codes.txt", "w").write("\n".join(sorted(codes)) + "\n")

In [ ]:
# Checkpoints in a private working folder (holds timestamps; deleted at the end)
WORK = f"{dest}/work_order"
os.makedirs(WORK, exist_ok=True)
def sh(cmd):
    r = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)
    out = r.stdout + r.stderr
    print("\n".join(l for l in out.splitlines() if l.startswith(("games read:", "check", "rows:", "games kept", "games per"))))
    if r.returncode:
        print("---- error output ----\n" + out[-4000:])
        raise RuntimeError("this step failed; the error text is above")
print("checkpoints so far:", sorted(os.listdir(WORK)) or "none")

In [ ]:
# Step 1: headers only, per game type and month
for t in TYPES:
    for m in MONTHS:
        f = f"scan_{t}_{m}.csv"
        if os.path.exists(os.path.join(WORK, f)):
            shutil.copy(os.path.join(WORK, f), f); print(t, m, "restored from checkpoint"); continue
        print("reading headers:", t, m)
        sh(f"curl -sL {url(t, m)} | zstdcat 2>/dev/null | python game_order.py scan --codes codes.txt --type {t} --out {f}")
        shutil.copy(f, WORK)

In [ ]:
# Step 2: build the order file and check that the numbering matches the variants files
sh(f"python game_order.py build --months {' '.join(MONTHS)} --out game_order_2018-06_2018-07.parquet --check-dir {dest}")
import pandas as pd
o = pd.read_parquet("game_order_2018-06_2018-07.parquet")
n = o.drop_duplicates(["variant", "game"]).groupby("variant").size()
for t in TYPES:
    print(f"{t:11s} games: {n.get(t, 0):,} (variants run kept {EXPECTED[t]:,})")

In [ ]:
# Safety check, then save to Drive and delete the working folder
ok_cols = ["player", "variant", "game", "seq", "gap_min", "prev_variant", "session", "session_pos", "base", "inc"]
assert list(o.columns) == ok_cols, "unexpected columns"
assert o.player.str.fullmatch(r"p_[0-9a-f]{12}").all(), "a player value is not an anonymous code"
assert set(o.variant) <= set(TYPES) and set(o.prev_variant.dropna()) <= set(TYPES)
print("safety check passed: anonymous codes only; no dates, no names")
shutil.copy("game_order_2018-06_2018-07.parquet", dest)
shutil.rmtree(WORK, ignore_errors=True)
print("saved to Drive:", f"{dest}/game_order_2018-06_2018-07.parquet", f"({os.path.getsize('game_order_2018-06_2018-07.parquet')/1e6:.1f} MB)")
print("working folder with timestamps deleted:", not os.path.exists(WORK))